In [1]:
!pip install git+https://github.com/facebookresearch/sam2.git

  Cloning https://github.com/facebookresearch/sam2.git to /tmp/pip-req-build-rs72lrf0
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/sam2.git /tmp/pip-req-build-rs72lrf0
  Resolved https://github.com/facebookresearch/sam2.git to commit 2b90b9f5ceec907a1c18123530e92e794ad901a4
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 1.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 5.0 MB/s eta 0:00:00
  Created wheel for SAM-2: filename=sam_2-1.0-cp312-cp312-linux_x86_64.whl size=504963 sha256=36900addfa7a73b668d0869f58fe2aafa46c283e20fa3d4797a353805dcddcff
  Stored in directory: /tmp/pip-ephem-wheel-cache-qvdyawd4/wheels/25/a3/8a/abd69dc6a6926b5e75c24810afac36c7b49b5c0f8a100147d6
  Created wheel for iopath: filename=iopath-0.1.10-py3-non

In [2]:
!wget -q https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_small.pt

In [3]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy.spatial.distance import directed_hausdorff
from PIL import Image

# Grounding DINO
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

# SAM 2
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

# ================= CONFIGURATION =================
IMG_DIR = "/kaggle/input/datasets/gonoszgonosz/rodent-data-2/processed/images"
MASK_DIR = "/kaggle/input/datasets/gonoszgonosz/rodent-data-2/processed/masks"

# Grounding DINO Prompt
TEXT_PROMPT = "rat."
BOX_THRESHOLD = 0.35
TEXT_THRESHOLD = 0.25

# SAM 2 Settings
SAM2_CHECKPOINT = "sam2_hiera_small.pt"
MODEL_CFG = "sam2_hiera_s.yaml"
OUTPUT_CSV = "/kaggle/working/grounded_sam2_metrics.csv"

BOUNDARY_DILATION = 7 
device = "cuda" if torch.cuda.is_available() else "cpu"
# =================================================

def calculate_iou(pred_mask, gt_mask):
    intersection = np.logical_and(pred_mask, gt_mask).sum()
    union = np.logical_or(pred_mask, gt_mask).sum()
    if union == 0: return 1.0 if intersection == 0 else 0.0
    return intersection / union

def calculate_dice(pred_mask, gt_mask):
    intersection = np.logical_and(pred_mask, gt_mask).sum()
    total = pred_mask.sum() + gt_mask.sum()
    if total == 0: return 1.0
    return (2. * intersection) / total

def calculate_boundary_iou(pred_mask, gt_mask, dilation=7):
    kernel = np.ones((dilation, dilation), dtype=np.uint8)
    gt_boundary = cv2.morphologyEx(gt_mask.astype(np.uint8), cv2.MORPH_GRADIENT, kernel) > 0
    pred_boundary = cv2.morphologyEx(pred_mask.astype(np.uint8), cv2.MORPH_GRADIENT, kernel) > 0
    
    intersection = np.logical_and(pred_boundary, gt_boundary).sum()
    union = np.logical_or(pred_boundary, gt_boundary).sum()
    
    if union == 0: return 1.0 if intersection == 0 else 0.0
    return intersection / union

def calculate_hausdorff(pred_mask, gt_mask):
    pred_edges = cv2.Canny((pred_mask.astype(np.uint8) * 255), 0, 1)
    gt_edges = cv2.Canny((gt_mask.astype(np.uint8) * 255), 0, 1)
    
    pred_pts = np.argwhere(pred_edges > 0)
    gt_pts = np.argwhere(gt_edges > 0)
    
    if len(pred_pts) == 0 or len(gt_pts) == 0:
        return np.nan 
        
    d1 = directed_hausdorff(pred_pts, gt_pts)[0]
    d2 = directed_hausdorff(gt_pts, pred_pts)[0]
    return max(d1, d2)

def main():
    print("--- LOADING GROUNDING DINO ---")
    gd_processor = AutoProcessor.from_pretrained("IDEA-Research/grounding-dino-base")
    gd_model = AutoModelForZeroShotObjectDetection.from_pretrained("IDEA-Research/grounding-dino-base").to(device)
    gd_model.eval()

    print("--- LOADING SAM 2 ---")
    sam2_model = build_sam2(MODEL_CFG, SAM2_CHECKPOINT, device=device)
    sam_predictor = SAM2ImagePredictor(sam2_model)

    all_images = [f for f in os.listdir(IMG_DIR) if f.endswith(('.jpg', '.png'))]
    all_masks = [f for f in os.listdir(MASK_DIR) if f.endswith(('.jpg', '.png'))]
    
    img_map = {os.path.splitext(f)[0]: f for f in all_images}
    mask_map = {os.path.splitext(f)[0]: f for f in all_masks}
    common_ids = sorted(list(set(img_map.keys()) & set(mask_map.keys())))

    evaluation_data = []
    
    print(f"--- STARTING GROUNDED-SAM 2 EVALUATION ON {len(common_ids)} FRAMES ---")
    pbar = tqdm(common_ids)
    
    for cid in pbar:
        img_path = os.path.join(IMG_DIR, img_map[cid])
        mask_path = os.path.join(MASK_DIR, mask_map[cid])
        
        image = cv2.imread(img_path)
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        pil_image = Image.fromarray(image_rgb)
        
        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_binary = np.where(gt_mask > 0, 1, 0).astype(bool)
        
        # 1. Grounding DINO Inference
        inputs = gd_processor(images=pil_image, text=TEXT_PROMPT, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = gd_model(**inputs)
        
        results = gd_processor.post_process_grounded_object_detection(
            outputs,
            inputs.input_ids,
            threshold=BOX_THRESHOLD,
            text_threshold=TEXT_THRESHOLD,
            target_sizes=[pil_image.size[::-1]]
        )[0]
        
        # 2. Pass Box to SAM 2
        if len(results["boxes"]) == 0:
            sam_pred_binary = np.zeros_like(gt_binary) # DINO failed to find the rat entirely
        else:
            best_box_idx = torch.argmax(results["scores"]).item()
            bbox = results["boxes"][best_box_idx].cpu().numpy()
            
            sam_predictor.set_image(image_rgb)
            masks, scores, logits = sam_predictor.predict(
                point_coords=None,
                point_labels=None,
                box=bbox[None, :],
                multimask_output=True, 
            )
            
            best_mask_idx = np.argmax(scores)
            sam_pred_binary = masks[best_mask_idx]
        
        # 3. Calculate metrics
        iou = calculate_iou(sam_pred_binary, gt_binary)
        dice = calculate_dice(sam_pred_binary, gt_binary)
        bound_iou = calculate_boundary_iou(sam_pred_binary, gt_binary, BOUNDARY_DILATION)
        hausdorff = calculate_hausdorff(sam_pred_binary, gt_binary)
        
        evaluation_data.append({
            "Frame_ID": cid,
            "mIoU": iou,
            "Dice": dice,
            "Boundary_IoU": bound_iou,
            "Hausdorff_Distance": hausdorff
        })
        
        pbar.set_postfix({"Avg B-IoU": f"{np.mean([x['Boundary_IoU'] for x in evaluation_data]):.4f}"})

    df = pd.DataFrame(evaluation_data)
    df.to_csv(OUTPUT_CSV, index=False)

    print("\n" + "="*50)
    print(" GROUNDED-SAM 2 ZERO-SHOT EVALUATION COMPLETED")
    print("="*50)
    print(f" Mean mIoU              : {df['mIoU'].mean():.4f}")
    print(f" Mean Dice Score        : {df['Dice'].mean():.4f}")
    print(f" Mean Boundary IoU      : {df['Boundary_IoU'].mean():.4f}")
    print(f" Mean Hausdorff Distance: {df['Hausdorff_Distance'].mean():.2f} pixels")
    print(f" Data saved to          : {OUTPUT_CSV}")
    print("="*50)

if __name__ == "__main__":
    main()

--- LOADING GROUNDING DINO ---


preprocessor_config.json:   0%|          | 0.00/457 [00:00<?, ?B/s]

The image processor of type `GroundingDinoImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/933M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1206 [00:00<?, ?it/s]

--- LOADING SAM 2 ---
--- STARTING GROUNDED-SAM 2 EVALUATION ON 306 FRAMES ---


100%|██████████| 306/306 [03:31<00:00,  1.45it/s, Avg B-IoU=0.1646]


 GROUNDED-SAM 2 ZERO-SHOT EVALUATION COMPLETED
 Mean mIoU              : 0.3976
 Mean Dice Score        : 0.4733
 Mean Boundary IoU      : 0.1646
 Mean Hausdorff Distance: 76.27 pixels
 Data saved to          : /kaggle/working/grounded_sam2_metrics.csv
